# SITKit Quickstart / 快速入门

Analyze one sentence with a local **Qwen3-8B or Llama3-8B base model**, with an optional reference model for comparison.  
使用一个本地 **Qwen3-8B 或 Llama3-8B 基础模型**分析一句文本，也可以添加可选的对标模型。

## 1. Install and import / 安装与导入

Install the official wheel before starting Jupyter.  
启动 Jupyter 前请先安装官方 wheel。

In [ ]:
# Run once if SITKit is not installed in this kernel:
# %pip install /path/to/sitkit-0.1.0-*.whl

from pathlib import Path
import json
import os

from IPython.display import Image, display
from sitkit import SurrogateInteractionExplainer, inspect_model

## 2. Configure local models / 配置本地模型

Set `MODEL` before launching Jupyter. `REFERENCE_MODEL` is optional.  
启动 Jupyter 前设置 `MODEL`；`REFERENCE_MODEL` 为可选的对标模型。

In [ ]:
MODEL = Path(os.environ.get("MODEL", "/models/Qwen3-8B"))
reference_value = os.environ.get("REFERENCE_MODEL")
REFERENCE_MODEL = Path(reference_value) if reference_value else None
INPUT_JSON = Path(os.environ.get("SITKIT_INPUT_JSON", "examples/datasets/economics.json"))
RUN_DIR = Path(os.environ.get("SITKIT_RUN_DIR", "runs/notebook-quickstart"))
RUN_DIR.mkdir(parents=True, exist_ok=True)
MODEL, REFERENCE_MODEL, INPUT_JSON, RUN_DIR

## 3. Check model admission / 检查模型兼容性

The architecture check is read-only. Every configured model must match a bundled official baseline.  
结构检查是只读操作，每个已配置模型都必须匹配随包提供的官方 baseline。

In [ ]:
checks = {"target_model": inspect_model(MODEL)}
if REFERENCE_MODEL is not None:
    checks["reference_model"] = inspect_model(REFERENCE_MODEL)
for role, check in checks.items():
    print(role, json.dumps(check.to_dict(), ensure_ascii=False, indent=2))
    if not check.ok:
        raise ValueError(f"{role} is not supported: {check.errors}")

## 4. Explain one sentence / 分析一个句子

No baseline path is needed. SITKit resolves and verifies the official baseline automatically.  
无需填写 baseline 路径；SITKit 会自动匹配并校验官方 baseline。

In [ ]:
samples = json.loads(INPUT_JSON.read_text(encoding="utf-8"))
sample = samples[0]

explainer = SurrogateInteractionExplainer(
    MODEL,
    reference_model=REFERENCE_MODEL,
    max_players=10,
    batch_size=16,
    device="auto",
    reward_type="gt",
)

result = explainer.explain(
    sample["text"],
    players=sample["players"],
    sample_id="economics-0001",
    target_text=sample.get("target_text"),
    top_k=5,
    output_dir=RUN_DIR,
)
payload = result.to_dict()
print(json.dumps(payload["models"], ensure_ascii=False, indent=2)[:6000])

## 5. Inspect top interactions / 查看主要交互

Positive and negative interactions are returned for every configured model.  
每个已配置模型都会分别返回正向和负向交互。

In [ ]:
for role in payload["models"]:
    print(f"\n{role} positive TOP 5")
    for row in payload["models"][role]["positive_top"]:
        print(row["type"], row["player_ids"], row["value"])
    print(f"{role} negative TOP 5")
    for row in payload["models"][role]["negative_top"]:
        print(row["type"], row["player_ids"], row["value"])

## 6. View report figures / 查看报告图

Every successful run produces the full report set: optimization loss, signed strength by order, AND/OR tree, salient interactions, and sparsity.  
每次成功运行都会生成完整报告：优化 loss、阶数强度、AND/OR 树、显著交互表和稀疏性图。

In [ ]:
for role in payload["models"]:
    report_dir = RUN_DIR / "economics-0001" / "report" / role
    print((report_dir / "inference.txt").read_text(encoding="utf-8"))
    print(role, report_dir / "salient_interaction.csv")
    for name in ("optimization_loss.png", "signed_strength_by_order.png", "and_or_tree.png", "sparsity.png"):
        display(Image(filename=str(report_dir / name)))

## Reproducibility / 可复现性

Keep the SITKit version, wheel checksum, model fingerprints, configuration, `inference.txt`, and report files together.  
请一并保存 SITKit 版本、wheel 校验值、模型指纹、运行参数、`inference.txt` 和报告文件。